## Complete Dictionary with included events as desired - May be exported for one time completion 

In [1]:
import os
import re
import pathlib as pl
import collections

In [2]:
os.chdir('..')
os.chdir('..')
home = pl.Path(os.getcwd())

#user to set variables for the project. tagged as parameter for papermill runs
project = 'wy_fy23'

In [3]:
print('home is at: ',home)
home = pl.Path(home)
from src.hdf import *

inputs = home/'inputs'
outputs_base = home/'outputs'

export_folder = outputs_base/project/'event_tie_ins'

home is at:  c:\_code\hms_to_ras_sst


Singular input expected below

In [4]:
#provide a list with all HUC10s that do not require tie-in review. ex. the model domain ends at a dam.
tie_in_exceptions = ['1008001203','1008000506','1008001006']

In [5]:
#create folder to export the tie-in dictionaries.
if not os.path.exists(export_folder):
    os.makedirs(export_folder)

In [6]:
with open(inputs/project/'dictionaries'/'HUC10_outflow_toHUC10.json') as src:
    huc_connect_huc = json.load(src)
with open(inputs/project/'dictionaries'/'HUC10_into_dsJunction.json') as src:
    huc_connect_j = json.load(src)

In [7]:
#list of model domains expected.
huc_list = ['1008001006', '1008001003','1008000506', '1008000504', '1008000505',
            '1008001203', '1008001202', '1008001303', '1008001302','1008001301'] #'1008001405', '1008000604'
# huc_list = ['1008001006','1008001405', '1008000604']
# huc_list = list(huc_connect_huc.keys())

Three following cells, each export a json file related to the model events

In [8]:
#dictionary to hold all recurrence interval, event and flow information together for all junctions in each model domain (according to their geojsons).
ALL_model_recur_event_flows = {}

for huc in huc_list:
    geojson_paths = glob.glob(str(inputs/project/'hms_ras'/huc/'*all_junctions_events.geojson'))

    #attain all recurrence interval information
    for path in geojson_paths:
        basename = pl.Path(path).stem
        recur_find = re.search(r'_[\d.]+[_mp]', basename)
        recur = recur_find.group(0).replace('_','')

        #identify all events related to this recurrence interval
        with open(path, 'r') as f:
            geojson_data = json.load(f)
        key_list = list(geojson_data['features'][0]['properties'].keys())
        event_matches = [item for item in key_list if re.search(r'P[\d]+_R[EY\-\d]+', item, re.IGNORECASE)]
        
        #iterate through each junction to collect information related to it.
        for i in range(0,len(geojson_data['features'])):
            j_name = (geojson_data['features'][i]['properties']['name'])
            j_events = [key for key in list(geojson_data['features'][0]['properties'].keys()) if key in event_matches]
            assert len(j_events) == len(event_matches), "Mismatch in number of events found"
            
            #in addition to flow per event at junction, the max flow event is also identified and stored.
            temp_event_flow = {}
            for event in j_events:
                ALL_model_recur_event_flows.setdefault(huc, {}).setdefault(j_name, {}).setdefault(recur, {}).setdefault('events', {})[event] = geojson_data['features'][i]['properties'][event]
                temp_event_flow.update({event: geojson_data['features'][i]['properties'][event]})
            max_flow_per_recur = max(temp_event_flow, key=temp_event_flow.get)
            ALL_model_recur_event_flows[huc][j_name][recur].update({'max_flow_event':{max_flow_per_recur:geojson_data['features'][i]['properties'][max_flow_per_recur]}})

with open(export_folder/"all_model_event_data.json", "w") as f:
    json.dump(ALL_model_recur_event_flows, f, indent=4)

In [9]:
#export a dictionary with each model domain, the recurrence interval and the events expected.
all_model_rec_events = {}
for huc in huc_list:
    all_model_rec_events[huc] = {}
    #use first junction to attain all recurrence intervals and events. Assumes all junctions have same events for each recurrence interval.
    j = list(ALL_model_recur_event_flows[huc].keys())[0]
    rec_intervals = list(ALL_model_recur_event_flows[huc][j].keys())
    all_model_rec_events[huc] = {}
    for r in rec_intervals:
        # print(list(ALL_model_recur_event_flows[huc][j][r]['events'].keys()))
        all_model_rec_events[huc][r] = list(ALL_model_recur_event_flows[huc][j][r]['events'].keys())

with open(export_folder/"all_model_rec_events.json", "w") as f:
    json.dump(all_model_rec_events, f, indent=4)

In [10]:
#dictionary to hold all downstream model tie-in information for all junctions. These are specific to models that require tie ins.
all_model_ds_data = {}

for huc in huc_list:
    print(huc)
    if huc in tie_in_exceptions:
        print('tie in event information will not be available for', huc)
    else:
        if huc_connect_huc[huc] != 'OUT' and huc_connect_j[huc] != 'N\A':
            dsj = huc_connect_j[huc]
            dshuc = huc_connect_huc[huc]

            for recur_int in ALL_model_recur_event_flows[huc][dsj].keys():
                tie_in_event = list(ALL_model_recur_event_flows[huc][dsj][recur_int]['max_flow_event'].keys())[0]
                tie_in_event_flow = list(ALL_model_recur_event_flows[huc][dsj][recur_int]['max_flow_event'].values())[0]

                upper_limit_event = list(ALL_model_recur_event_flows[dshuc][dsj][recur_int]['max_flow_event'].keys())[0]
                upper_limit_flow = list(ALL_model_recur_event_flows[dshuc][dsj][recur_int]['max_flow_event'].values())[0]

                us_ds_flow_ratio = round(float(tie_in_event_flow)/float(upper_limit_flow),3)

                events_exceed = {k:v for k,v in ALL_model_recur_event_flows[huc][dsj][recur_int]['events'].items() if v > upper_limit_flow and k != tie_in_event}
                if events_exceed:
                    max_event_events_exceeded = max(events_exceed, key=events_exceed.get)
                    max_flow_events_exceeded = max(events_exceed.values())
                    max_flow_ratio_to_dshuc = round(float(max_flow_events_exceeded)/float(upper_limit_flow),3)
                else:
                    max_event_events_exceeded = None
                    max_flow_events_exceeded = None
                    max_flow_ratio_to_dshuc = None
                all_model_ds_data.setdefault(huc, {}).setdefault(dsj, {}).setdefault(recur_int, {}).update({'ds_huc': dshuc,
                                                                                                            'model_tie_in_event': tie_in_event,
                                                                                                            'model_tie_in_flow': tie_in_event_flow,
                                                                                                            'ds_tie_in_event': upper_limit_event,
                                                                                                            'ds_tie_in_flow': upper_limit_flow,
                                                                                                            'us_ds_flow_ratio': us_ds_flow_ratio,
                                                                                                            'us_ds_flow_ratio_flagged': True if us_ds_flow_ratio >= 1.2 or us_ds_flow_ratio <= 0.8 else False, #This is the threshold of 20% difference prior to flagging
                                                                                                            'us_events_exceeding_ds_tie_in': events_exceed,
                                                                                                            'max_event_from_exceeding': max_event_events_exceeded,
                                                                                                            'max_flow_from_exceeding': max_flow_events_exceeded,
                                                                                                            'max_flow_ratio_us_ds': max_flow_ratio_to_dshuc,
                                                                                                            'max_flow_ratio_flagged': True if max_flow_ratio_to_dshuc != None and max_flow_ratio_to_dshuc >= 1.2 else False})
with open(export_folder/"all_model_ds_event_data.json", "w") as f:
    json.dump(all_model_ds_data, f, indent=4)

1008001006
tie in event information will not be available for 1008001006
1008001003
1008000506
tie in event information will not be available for 1008000506
1008000504
1008000505
1008001203
tie in event information will not be available for 1008001203
1008001202
1008001303
1008001302
1008001301


In [11]:
#Finished